# Example: Production Portfolio Engine

In this example, we build the full production loop: the bandit selects assets daily, the Cobb-Douglas allocator decides how much of each, sentiment monitoring adjusts lambda in real time, and escalation triggers provide the human safety net. We simulate 60 trading days on an HMM-generated path with a regime shift mid-period. Let's dive in!

> **By the end of this example, you will be able to:**
> * __Run the production loop:__ Execute the bandit and Cobb-Douglas allocator together over a simulated trading period. Observe how the system rebalances daily in response to market conditions.
> * __Implement sentiment monitoring:__ Configure automatic lambda override when sentiment deteriorates below a threshold. Evaluate how the override shifts the portfolio toward safer assets.
> * __Define human escalation triggers:__ Set up escalation rules for drawdown, sentiment, and bandit churn events. Inspect the audit trail that these triggers produce during stress events.

## Setup, Data and Prerequisites
We begin by loading packages and defining the simulation constants. The code cell below sets global parameters (time step, risk-free rate, starting capital, EMA windows, ticker list, and SIM parameters) and stores them in global variables.

In [ ]:
include("Include.jl");

In [ ]:
let
    # --- Global simulation constants ---
    global Δt = 1.0 / 252.0;           # daily time step (trading year = 252 days)
    global rf = 0.05;                   # annualized risk-free rate
    global B₀ = 10000.0;               # starting capital ($)
    global N_short = 21;                # short EMA window (≈ 1 month)
    global N_long = 63;                 # long EMA window (≈ 1 quarter)
    global N_growth = 10;               # growth-rate EMA window
    global GAIN = 10.0;                 # sigmoid gain for lambda computation
    global offset = N_short + N_long;   # burn-in days required before production starts
    global n_production_days = 60;      # number of production trading days to simulate
    global T_total = offset + n_production_days + 20; # total path length (with buffer)

    # --- Ticker universe and Single-Index Model (SIM) parameters ---
    # Each ticker maps to (alpha, beta, idiosyncratic_vol)
    global tickers = ["LargeCap", "SmallCap", "International", "Bond", "Commodity"];
    global sim_params = Dict(
        "LargeCap"      => (0.0002, 1.10, 0.010),
        "SmallCap"      => (0.0003, 1.35, 0.014),
        "International" => (0.0001, 0.95, 0.012),
        "Bond"          => (0.0001, -0.15, 0.003),
        "Commodity"     => (0.0001, 0.60, 0.013)
    );

    # --- Starting prices for each ticker ---
    global start_prices = Dict("LargeCap" => 150.0, "SmallCap" => 45.0,
        "International" => 80.0, "Bond" => 100.0, "Commodity" => 60.0);

    println("Setup: $(n_production_days) production days, $(length(tickers)) tickers")
end

Next, we generate a single HMM path with synthetic per-ticker prices for the production simulation. The code below fits an HMM, simulates one market path, constructs per-ticker prices via the Single-Index Model, computes EMA and lambda series, and generates synthetic sentiment. The results are stored in the `market_prices::Vector{Float64}`, `price_matrix::Matrix{Float64}`, `lambda_series::Vector{Float64}`, and `sentiment_series::Vector{Float64}` variables.

In [ ]:
let
    # --- Step 1: Fit an HMM on synthetic training prices ---
    training = generate_training_prices(S₀=100.0, μ=0.08, σ=0.18, T=1260, seed=42);
    global hmm_model = hmm_fit(JumpHiddenMarkovModel, training; rf=rf, N=50, nu=5.0, dt=Δt);
    global hmm_model = hmm_tune(hmm_model, training;
        epsilon_range=range(1e-4, 2.5e-2, length=8),
        lambda_range=range(10.0, 80.0, length=6), n_paths=30);

    # --- Step 2: Simulate one market-level price path ---
    result = hmm_simulate(hmm_model, T_total; n_paths=1);
    G = result.paths[1].observations;                                       # growth rates
    global market_prices = JumpHMM.prices_from_growth_rates(G, 100.0;       # convert to prices
        rf=rf, dt=Δt);

    # --- Step 3: Generate per-ticker prices via Single-Index Model (SIM) ---
    K = length(tickers);
    global price_matrix = zeros(T_total, K + 1);   # col 1 = day index, cols 2:end = ticker prices
    price_matrix[:, 1] = 1:T_total;
    gm_raw = compute_market_growth(market_prices; Δt=Δt);  # daily market growth rates

    for (k, ticker) in enumerate(tickers)
        (αᵢ, βᵢ, σᵢ) = sim_params[ticker];        # unpack SIM parameters
        price_matrix[1, k + 1] = start_prices[ticker];
        for t in 2:T_total
            gm_t = gm_raw[min(t - 1, length(gm_raw))];    # market growth at time t
            gᵢ = αᵢ + βᵢ * gm_t * Δt + σᵢ * sqrt(Δt) * randn();  # SIM return
            price_matrix[t, k + 1] = price_matrix[t - 1, k + 1] * exp(gᵢ);
        end
    end

    # --- Step 4: Compute EMA series and lambda (risk-aversion signal) ---
    global ema_short = compute_ema(market_prices; window=N_short);
    global ema_long = compute_ema(market_prices; window=N_long);
    global lambda_series = compute_lambda(ema_short, ema_long; G=GAIN);
    lambda_series[1:offset] .= 0.0;    # zero out burn-in period
    global gm_ema = compute_ema(gm_raw; window=N_growth);

    # --- Step 5: Generate synthetic sentiment signal ---
    global sentiment_series = generate_synthetic_sentiment(market_prices; noise_σ=0.15, seed=77);

    println("Generated $(T_total)-day path with sentiment")
end

___
## Task 1: Build and Run the Production Loop
We define the production context (risk parameters, escalation thresholds) and run the full production simulation for 60 trading days. The bandit selects assets daily, the Cobb-Douglas allocator computes the optimal allocation, and trigger rules enforce safety.

> __What should you see?__
>
> A wealth curve that responds to market conditions. The bandit may change its selected assets as conditions shift. The system should rebalance most days but may hold or de-risk when escalation triggers fire.

The code below builds a `MyProductionContext` and passes it to [the `run_production_simulation(...)` function](../../code/docs/build/session4.html). The results are stored in the `prod_results::Vector` and `prod_events::Vector` variables.

In [ ]:
let
    # --- Step 1: Build the production context with risk and escalation parameters ---
    global prod_ctx = build(MyProductionContext, (
        tickers = tickers,
        sim_parameters = sim_params,
        B₀ = B₀,
        epsilon = 0.1,                     # bandit exploration rate
        max_drawdown = 0.15,               # escalation trigger: max drawdown threshold
        max_turnover = 0.50,               # escalation trigger: max daily turnover
        sentiment_threshold = -0.5,        # sentiment level that triggers lambda override
        sentiment_override_lambda = 2.0,   # very risk-averse lambda when sentiment crashes
        max_bandit_churn = 2               # max allowed asset changes per day
    ));

    # --- Step 2: Run the production simulation for n_production_days ---
    println("Running $(n_production_days)-day production simulation...")
    global (prod_results, prod_events) = run_production_simulation(
        prod_ctx, market_prices, price_matrix,
        sentiment_series, lambda_series, gm_ema;
        n_days=n_production_days, offset=offset, n_bandit_iters=100
    );

    # --- Step 3: Print summary statistics ---
    println("Simulation complete:")
    println("  Final wealth: \$$(round(prod_results[end].wealth, digits=2))")
    println("  Escalation events: $(length(prod_events))")
    println("  Days rebalanced: $(sum(r.rebalanced for r in prod_results))")
end

The code below plots the production wealth curve with escalation markers overlaid as vertical lines.

In [ ]:
let
    # --- Step 1: Extract day and wealth vectors from production results ---
    days = [r.day for r in prod_results];
    wealth = [r.wealth for r in prod_results];

    # --- Step 2: Plot the wealth curve ---
    p = plot(days, wealth, label="Production Wealth", linewidth=2.5, color=:steelblue,
        xlabel="Production Day", ylabel="Wealth (\$)",
        title="Production Portfolio Engine", size=(800, 400), legend=:topright)
    hline!(p, [B₀], label="Starting Capital", linestyle=:dash, color=:gray50, alpha=0.5)

    # --- Step 3: Overlay escalation events as vertical lines ---
    for evt in prod_events
        if evt.day <= n_production_days
            color = evt.severity == :critical ? :red : :orange;
            vline!(p, [evt.day], label="", color=color, alpha=0.3, linewidth=2)
        end
    end

    # --- Step 4: Add legend entries for escalation severity levels ---
    if any(e.severity == :critical for e in prod_events)
        plot!(p, [], [], label="Critical Escalation", color=:red, linewidth=2)
    end
    if any(e.severity == :warning for e in prod_events)
        plot!(p, [], [], label="Warning Escalation", color=:orange, linewidth=2)
    end
    p
end

___
## Task 2: Sentiment Monitoring with Lambda Override
We visualize how the sentiment signal interacts with the production system. When sentiment drops below the threshold ($s_t < -0.5$), the system overrides lambda to a high risk-aversion value, causing the allocator to shift toward safer assets.

> __What should you see?__
>
> The sentiment time series should correlate with market movements (positive during rallies, negative during drops). When it crosses the threshold, the effective lambda jumps up, and the portfolio response should be visible as reduced risk exposure.

The code below constructs a three-panel figure showing sentiment, effective lambda, and portfolio wealth over the production period.

In [ ]:
let
    # --- Step 1: Extract time series from production results ---
    days = [r.day for r in prod_results];
    sentiments = [r.sentiment for r in prod_results];
    lambdas = [r.lambda for r in prod_results];
    wealth = [r.wealth for r in prod_results];

    # --- Step 2: Top panel - sentiment score with threshold ---
    p1 = plot(days, sentiments, label="Sentiment", linewidth=2, color=:purple)
    hline!(p1, [prod_ctx.sentiment_threshold], label="Threshold", linestyle=:dash, color=:red)
    hline!(p1, [0.0], label="", linestyle=:dot, color=:black, alpha=0.3)
    ylabel!(p1, "Sentiment Score")
    title!(p1, "Sentiment Monitoring")

    # --- Step 3: Middle panel - effective lambda (with override visible) ---
    p2 = plot(days, lambdas, label="Effective λ", linewidth=2, color=:coral)
    ylabel!(p2, "λ (risk-aversion)")
    title!(p2, "Lambda (with sentiment override)")

    # --- Step 4: Bottom panel - portfolio wealth response ---
    p3 = plot(days, wealth, label="Wealth", linewidth=2, color=:steelblue)
    xlabel!(p3, "Production Day")
    ylabel!(p3, "Wealth (\$)")
    title!(p3, "Portfolio Response")

    plot(p1, p2, p3, layout=(3, 1), size=(800, 700), legend=:topright)
end

___
## Task 3: Human Override Rules and Escalation Log
We examine every escalation event that fired during the simulation, including what triggered it, how severe it was, and what the system recommended. This is the audit trail that the investment committee reviews.

> __What should you see?__
>
> A table of escalation events with timestamps, trigger types, severity levels, and recommended actions. Critical events should correspond to large drawdowns. Warnings should correspond to sentiment crashes or bandit churn.

The code below uses [the `check_escalation_triggers(...)` function](../../code/docs/build/session4.html) logic to format the escalation log as a markdown table via `pretty_table`.

In [ ]:
let
    # --- Display the escalation event log as a formatted table ---
    if length(prod_events) > 0
        println("═"^70)
        println("  ESCALATION LOG: $(length(prod_events)) events")
        println("═"^70)

        # Build a DataFrame from the escalation event structs
        escalation_df = DataFrame(
            "Day" => [e.day for e in prod_events],
            "Trigger" => [e.trigger_type for e in prod_events],
            "Severity" => [string(e.severity) for e in prod_events],
            "Message" => [e.message for e in prod_events],
            "Recommendation" => [e.recommended_action for e in prod_events]
        );
        pretty_table(escalation_df, tf=tf_markdown,
            columns_width=[5, 16, 10, 40, 40])
    else
        println("No escalation events during this simulation.")
        println("The market path was calm enough that no triggers fired.")
    end
end

Next, we visualize the bandit's selected asset subset over time. The code below builds a scatter plot showing which assets the bandit included on each production day.

In [ ]:
let
    K = length(tickers);
    days = [r.day for r in prod_results];
    n = length(days);

    # --- Step 1: Build inclusion matrix (1 = selected, 0 = not selected) ---
    inclusion = zeros(n, K);
    for (i, r) in enumerate(prod_results)
        inclusion[i, :] = r.bandit_action;
    end

    # --- Step 2: Scatter plot with one row per ticker ---
    colors = [:steelblue :coral :green :goldenrod :purple];
    p = plot(size=(800, 300), title="Bandit Asset Selection Over Time",
        xlabel="Production Day", ylabel="", yticks=(1:K, tickers), legend=false)

    for k in 1:K
        for i in 1:n
            if inclusion[i, k] == 1
                scatter!(p, [days[i]], [k], markersize=4, color=colors[k], alpha=0.7)
            end
        end
    end
    p
end

Finally, the code below computes and prints summary metrics for the production run using [the `compute_dashboard_metrics(...)` function](../../code/docs/build/session4.html), then saves the results to disk.

In [ ]:
let
    # --- Step 1: Compute aggregate dashboard metrics ---
    metrics = compute_dashboard_metrics(prod_results, prod_events);

    # --- Step 2: Print the production summary ---
    println("═"^50)
    println("  PRODUCTION SUMMARY")
    println("═"^50)
    println("  Days simulated:       $(metrics["n_days"])")
    println("  Final wealth:         \$$(round(metrics["final_wealth"], digits=2))")
    println("  Max drawdown:         $(round(metrics["max_drawdown"]*100, digits=1))%")
    println("  Rebalance days:       $(metrics["n_rebalances"])")
    println("  Bandit subset changes: $(metrics["n_bandit_changes"])")
    println("  Escalations:          $(metrics["n_escalations"]) ($(metrics["n_critical"]) critical, $(metrics["n_warning"]) warning)")
    println("  Avg sentiment:        $(round(metrics["avg_sentiment"], digits=3))")

    # --- Step 3: Save results to disk for use in the next example ---
    save_production_results(joinpath(_PATH_TO_DATA, "production-results.jld2"),
        prod_results, prod_events);
end

___
## Summary
This example demonstrated a complete production portfolio engine that combines bandit asset selection, Cobb-Douglas allocation, sentiment monitoring, and human escalation triggers over a 60-day simulated trading period.

### Key Takeaways
* **Bandit and Cobb-Douglas allocator work together:** The bandit selects which assets to include each day, and the Cobb-Douglas allocator decides how much capital to assign to each selected asset. This combination provides adaptive asset selection with utility-based allocation.
* **Sentiment monitoring enables proactive de-risking:** When sentiment drops below the configured threshold, the system overrides lambda to a defensive value before drawdowns materialize. This mechanism shifts the portfolio toward safer assets without waiting for losses to accumulate.
* **Escalation triggers create an auditable safety net:** Every trigger event (drawdown breach, sentiment crash, bandit churn) is logged with severity, message, and recommended action. This audit trail gives the investment committee full transparency into autonomous decisions.

In the next example, we build an operational dashboard and stress-test the system with a crash injection.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice.